# Экономичные по производительности разреженные автоэнкодеры: Нулевая гибель нейронов и предсказуемый контроль дисперсии для данных с высокой энтропией

***Draft Materials Notebook***  

---

## 1. АННОТАЦИЯ 

Разреженные автоэнкодеры широко применяются в задачах механистической 
интерпретируемости [1] и позволяют достигать высокой степени разреженности 
(70–90%) на данных с низкой дисперсией. Однако их поведение при обучении 
на данных с высокой дисперсией остаётся недостаточно исследованным.

Мы обнаружили этот пробел, исследуя нейронные сети для генерации энтропии: 
при проверке криптографического прототипа было выявлено, что латентные 
представления автоэнкодера коллапсируют по дисперсии (порядка 10⁻¹²). 
Это послужило мотивацией для систематического эмпирического исследования 
потери вместимости в разреженных автоэнкодерах через четыре архитектурных 
итерации.

Результаты показали, что традиционные подходы систематически приводят к 
потере вместимости: регуляризация по L1 даёт полный коллапс нейронов, 
отбор верхних K элементов на основе ReLU приводит к 35% мёртвых нейронов 
при дисперсии 0.062.

Наше решение сочетает отбор верхних K элементов с функцией активации 
sin(8x)+0.5·tanh(4x). Систематическое тестирование шести функций активации 
при пяти независимых запусках каждой показало, что только предложенная 
архитектура достигает нулевого процента мёртвых нейронов при дисперсии 0.418 
(улучшение в 4.4 раза над tanh, p<0.0001). Метод обеспечивает монотонный 
контроль зависимости дисперсии от параметра разреженности K.

Тестирование выявило контринтуитивную находку: функция LeakyReLU даёт больше 
мёртвых нейронов (66) чем стандартная ReLU (45) из-за взаимодействия с 
механизмом отбора по абсолютному значению. Свойства метода подтверждены на 
четырёх доменах данных, включая систему Лоренца-96 и MNIST.

### 1.2 Постановка задачи: Потеря вместимости в разреженных автоэнкодерах

Данное исследование было мотивировано экспериментами с нейронными моделями, 
использующими автоэнкодеры для генерации высокоэнтропийных представлений в 
криптографических прототипах. Несмотря на корректное выполнение шифрования и 
дешифрования и высокую статистическую энтропию зашифрованных данных (7.59 бит 
на байт), анализ скрытого пространства выявил нетривиальные ограничения в 
структуре латентных представлений.

В частности, плотный автоэнкодер с 64-мерным латентным пространством, обученный 
на изображениях логистической карты, демонстрировал резкий коллапс дисперсии 
латентных признаков. Эмпирический анализ показал, что вариация по всем измерениям 
латентного пространства находилась на уровне порядка 10⁻¹², несмотря на 
использование нелинейных активаций.

Первоначально мы предположили, что специально подобранные функции активации 
могут сохранить хаотическую динамику в латентном пространстве, а именно 
экспоненциальную дивергенцию близких траекторий, характеризуемую положительными 
показателями Ляпунова. Эта гипотеза не подтвердилась: тест на дивергенцию в 
стиле Ляпунова показал соотношение около 1.07 при ожидаемом значении больше 5.0 
для экспоненциального роста. Автоэнкодер обрабатывает каждый кадр независимо и 
не имеет временной памяти, поэтому не может сохранять временной хаос.

Однако в процессе систематического тестирования была обнаружена более 
фундаментальная проблема. Разреженные автоэнкодеры теряют репрезентативную 
вместимость через два механизма. Первый — мёртвые нейроны: измерения латентного 
пространства перестают активироваться (от 35% при Top-K с ReLU до 100% при 
регуляризации по L1). Второй — коллапс дисперсии: латентное пространство 
сжимается (дисперсия 0.062 при 128 измерениях с ReLU).

Потеря вместимости указывает на фундаментальное ограничение нейронных представлений, 
критичное для мобильного машинного обучения, длительного обучения, представлений 
с сохранением приватности и генерации случайности.

Возможно ли разработать архитектуру разреженного автоэнкодера, которая минимизирует 
гибель нейронов, сохраняет высокую дисперсию при разреженности и обеспечивает 
предсказуемый контроль соотношения разреженности и дисперсии?

## 2. ВСТУПЛЕНИЕ

### 2.1 Разреженные автоэнкодеры в 2024-2025 годах

Разреженные автоэнкодеры получили широкое распространение как инструмент механистической интерпретации [1]. Основная идея заключается в изучении 
переполненных словарей интерпретируемых признаков путём обеспечения разреженности 
в скрытом пространстве, при этом разреженность обычно достигает от 70 до 90 
процентов при сохранении точности реконструкции. Недавняя работа компании 
Anthropic масштабировала разреженные автоэнкодеры до производственных языковых 
моделей [2], демонстрируя, что разреженные представления 
позволяют лучше понять поведение нейронной сети.

Однако эти достижения были сосредоточены в основном на статическом распределении 
данных или распределении с низкой дисперсией, например на векторных представлениях 
токенов в языковых моделях. Остаётся неисследованным важный вопрос: как разреженные 
автоэнкодеры ведут себя на данных с высокой дисперсией и сохраняют ли они 
репрезентативную вместимость?

### 2.2 Проблема: разреженность вызывает потерю вместимости

Данные с высокой дисперсией, такие как хаотические системы, естественные 
изображения и временные последовательности, характеризуются широким распределением 
значений в латентном пространстве, сложной внутренней структурой и требованиями к 
удержанию информации. Мы предположили, что стандартные методы разреживания будут 
систематически приводить к потере вместимости на таких данных. Наши предварительные 
эксперименты (раздел 1.2) подтвердили эту гипотезу: все протестированные стандартные 
методы привели к значительной потере вместимости через мёртвые нейроны, коллапс 
дисперсии, или оба механизма одновременно.

### 2.3 Наш вклад

Мы представляем систематическое исследование методом абляции разреженных 
автоэнкодеров на данных с высокой дисперсией через четыре архитектурных итерации, 
кульминацией которых является архитектура с отбором верхних K элементов и функцией 
активации sin(8x)+0.5·tanh(4x), демонстрирующая парето-оптимальность по критериям 
вместимости и дисперсии (подробные результаты в разделах 5–6).

Основные результаты работы:

- Систематическое тестирование шести функций активации при пяти независимых запусках 
каждой выявило, что только предложенная архитектура одновременно устраняет мёртвые 
нейроны и сохраняет высокую дисперсию латентного пространства (Таблица 1).

- Архитектура обеспечивает гладкий монотонный контроль зависимости дисперсии от 
числа активных элементов без единого нарушения тренда при значениях K от 4 до 112, 
что позволяет предсказуемо выбирать параметр K на этапе проектирования (Таблица 3).

- Тестирование методом абляции выявило контринтуитивную находку: функция LeakyReLU 
даёт больше мёртвых нейронов, чем стандартная ReLU, из-за взаимодействия с 
механизмом отбора по абсолютному значению (раздел 5.1).

- Все результаты получены при множественных запусках с t-тестами и доверительными 
интервалами. Воспроизводимость подтверждена на четырёх доменах данных (раздел 6).

Исходная гипотеза о сохранении временного хаоса не подтвердилась (раздел 5.7). 
Метод даёт увеличение ошибки реконструкции на 6.2 процента по сравнению с плотной 
базовой архитектурой из-за ограниченного диапазона выхода функции активации.

## 3. ЭКСПЕРИМЕНТАЛЬНАЯ УСТАНОВКА

### 3.1 Датасеты

Основной датасет генерируется из логистической карты x_{n+1} = r·x_n·(1 - x_n) 
с параметром r = 3.99, обеспечивающим хаотический режим. Каждое изображение 
размером 28×28 создаётся последовательной итерацией из случайного начального 
значения x_0 в интервале от 0 до 1. Обучающая выборка содержит 2000 изображений, 
валидационная 200, тестовая 500.

Для проверки обобщаемости используется карта Хенона с уравнениями x_{n+1} = 1 - a·x_n² + y_n 
и y_{n+1} = b·x_n при a=1.4, b=0.3. Размеры выборок соответствуют основному датасету.

### 3.2 Метрики оценки

Средняя дисперсия вычисляется как среднее арифметическое дисперсий по всем 
измерениям латентного пространства. Нейрон считается мёртвым если абсолютное 
значение его активации меньше 10^(-6) для всех образцов. Разреженность определяется 
как доля нулевых элементов после применения отбора верхних K элементов. 
Валидационная ошибка измеряется как среднеквадратичная ошибка между входными 
изображениями и их реконструкциями.

### 3.3 Статистический протокол

Тестирование функций активации методом абляции выполняется с пятью независимыми 
запусками для каждой функции. Сравнение с базовой архитектурой проводится с 
десятью запусками и вычислением 95-процентных доверительных интервалов. Проверка 
влияния регуляризатора целевой дисперсии выполняется с пятью запусками для каждого 
условия. Исследование зависимости от параметра K проводится с тремя запусками для 
каждого значения.

Статистическая значимость различий проверяется t-критерием Стьюдента в варианте 
Уэлча с порогом p < 0.05. Все результаты представлены как среднее значение плюс-минус 
стандартное отклонение по множественным запускам.

Обучение всех моделей выполняется в течение 10 эпох с размером пакета 64, используя 
оптимизатор Adam со скоростью обучения 0.001. Эти параметры зафиксированы для 
всех экспериментов.

**Эксперимент 1: Стандартный плотный релейный**

In [1]:
# Code: Standard Dense ReLU Autoencoder
def build_dense_relu_ae(image_size=(28, 28), latent_dim=64):
    h, w = image_size
    input_img = keras.Input(shape=(h, w, 1))
    x = layers.Flatten()(input_img)
    
    # Encoder
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    latent = layers.Dense(latent_dim, activation="relu", name="latent")(x)
    
    encoder = keras.Model(input_img, latent, name="dense_encoder")
    
    # Decoder
    x = layers.Dense(128, activation="relu")(latent)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(h * w, activation="sigmoid")(x)
    decoded = layers.Reshape((h, w, 1))(x)
    
    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    
    return autoencoder, encoder

**Результаты (Baseline Dense ReLU):**
```
Dead neurons: 47% even without explicit sparsity enforcement
Natural sparsity: ~35-40%
Variance: Moderate but unstable
Conclusion: ReLU naturally kills neurons on chaotic data
```

## 4. МЕТОД: EVOLUTION V1 → V4

### 4.1 Версия 1: Нарушена (L1 + регуляризация активности)

**Гипотеза:** Стандартные методы разреженного автоэнкодирования будут работать с хаотическими данными.

**Архитектура:**

In [ ]:
# V1: Standard Sparse AE with L1 + Activity regularization
def build_v1_broken_sparse_ae(latent_dim=128):
    # Encoder with heavy regularization
    latent = layers.Dense(
        latent_dim,
        activation="relu",
        activity_regularizer=keras.regularizers.l1(1e-4),  # Strong L1
        kernel_regularizer=keras.regularizers.l1(1e-4),
        name="latent"
    )(x)
    latent = layers.Dropout(0.2)(latent)  # Additional dropout

Первая попытка применить стандартные методы разреживания использовала 
регуляризацию по L1 с коэффициентом 0.0001 в сочетании с регуляризацией 
активности и прореживанием с вероятностью 0.2. Обучение на логистической 
карте привело к полному коллапсу: все 128 измерений латентного пространства 
перестали активироваться, дисперсия упала до нуля, разреженность достигла 
100 процентов. Ошибка валидации составила 0.116 что формально сопоставимо 
с работающими моделями, однако латентное пространство оказалось полностью 
нефункциональным. Стандартная агрессивная регуляризация несовместима с 
данными высокой дисперсии.

### 4.2 Версия 2: Простое исправление (только L2, без L1)

**Гипотеза:** Убрать регуляризацию, вызывающую разреженность, оставить только L2 для стабильности.

**Архитектура:**

In [ ]:
# V2: Simple fix - remove L1, keep only L2
def build_v2_simple_fix(latent_dim=128):
    latent = layers.Dense(
        latent_dim,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-4),  # L2 only
        name="latent"
    )(x)
    # No dropout, no L1

Удаление регуляризации по L1 с сохранением только регуляризации по L2 
восстановило частичную работоспособность. Около 90 из 128 нейронов стали 
активными, дисперсия поднялась до 0.013, ошибка валидации составила 0.114. 
Однако проблема мёртвых нейронов сохранилась на уровне 29 процентов, 
естественная разреженность около 29 процентов возникала без возможности 
контроля. Функция активации ReLU систематически приводила к гибели части 
нейронов даже без явной регуляризации разреженности.

### 4.3 Версия 3: K-разреженность с помощью ReLU (выбор Top-K)

**Гипотеза:** Использовать выбор Top-K [3] для явного управления разреженностью (K=32 из 128, то есть 75% разреженности).

**Архитектура:**

In [ ]:
# V3: K-Sparse with standard ReLU activation
@keras.saving.register_keras_serializable()
class TopKLayer(layers.Layer):
    def __init__(self, k=32, **kwargs):
        super().__init__(**kwargs)
        self.k = k

    def call(self, x):
        # Keep top-K activations, zero out the rest
        values, indices = tf.nn.top_k(x, k=self.k)
        mask = tf.reduce_sum(
            tf.one_hot(indices, depth=tf.shape(x)[-1]),
            axis=1
        )
        return x * mask

def build_v3_ksparse_relu(latent_dim=128, k_active=32):
    # Encoder
    latent_raw = layers.Dense(latent_dim, activation="relu")(x)
    latent = TopKLayer(k=k_active, name="topk")(latent_raw)  # 75% sparsity
    # K=32 out of 128 = 25% active = 75% sparse

Введение механизма отбора верхних K элементов позволило явно контролировать 
разреженность на уровне 75 процентов. При K равном 32 из 128 измерений 
ровно 32 нейрона остаются активными после каждого прохода. Ошибка валидации 
сохранилась на уровне 0.114. Однако проблема мёртвых нейронов усилилась: 
61 процент измерений (78 из 128) никогда не использовались, дисперсия 
составила 0.040. Механизм отбора верхних K в сочетании с ReLU создаёт 
положительную обратную связь: нейроны с малыми начальными весами попадают 
в отрицательную область ReLU, обнуляются, не получают градиентов и застревают 
в нулевом состоянии навсегда.

### 4.4 Версия 4: K-разреженный хаотический автоэнкодер (окончательное решение)

**Гипотеза:** Заменить ReLU на сохраняющую хаос активацию, которая имеет:
1. **Нет фиксированных точек** (позволяет избежать гибели нейронов)
2. **Высокая производная** везде (поддерживает градиентный поток)
3. **Расширяющаяся динамика** (сохраняет дисперсию)

#### Функция активации хаоса

Мы разработали пользовательскую активацию:

```python
chaos_activation(x) = sin(8x) + 0.5·tanh(4x)
```

**Architecture:**
**Свойства:**
- Производная: ~8·cos(8x) + 2·sech²(4x) ≈ 10 at typical scales
- Производная ReLU: 0 (x<0) или 1 (x>0) - намного меньше
- **Нет фиксированных точек** в соответствующей области
- Колебательный компонент (sin) обеспечивает хаотическое перемешивание
- Плавный компонент (tanh) обеспечивает стабильность

**Архитектура:**

In [ ]:
# V4: K-Sparse with Chaos Activation
@keras.saving.register_keras_serializable()
def chaos_activation(x):
    """Custom activation for chaos preservation"""
    return tf.sin(8.0 * x) + 0.5 * tf.tanh(4.0 * x)

@keras.saving.register_keras_serializable()
class TargetVarianceRegularizer(keras.regularizers.Regularizer):
    """Encourage latent variance to match target"""
    def __init__(self, target_variance=0.1, lambda_reg=0.01):
        self.target_variance = target_variance
        self.lambda_reg = lambda_reg

    def __call__(self, x):
        variance = tf.math.reduce_variance(x, axis=0)
        mean_variance = tf.reduce_mean(variance)
        penalty = tf.abs(mean_variance - self.target_variance)
        return self.lambda_reg * penalty

def build_v4_ksparse_chaos(latent_dim=128, k_active=32):
    h, w = image_size
    input_img = keras.Input(shape=(h, w, 1))
    x = layers.Flatten()(input_img)
    
    # Encoder with chaos activation
    x = layers.Dense(256)(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dense(128)(x)
    x = layers.Activation(chaos_activation)(x)
    
    # Latent with target variance regularizer
    latent_raw = layers.Dense(
        latent_dim,
        activity_regularizer=TargetVarianceRegularizer(
            target_variance=0.1,
            lambda_reg=0.01
        )
    )(x)
    latent_activated = layers.Activation(chaos_activation)(latent_raw)
    latent = TopKLayer(k=k_active, name="topk")(latent_activated)
    
    encoder = keras.Model(input_img, latent, name="chaos_encoder")
    
    # Decoder (symmetric)
    x = layers.Dense(128)(latent)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dense(256)(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dense(h * w, activation="sigmoid")(x)
    decoded = layers.Reshape((h, w, 1))(x)
    
    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="mse"
    )
    
    return autoencoder, encoder

Замена функции ReLU на комбинацию sin(8x)+0.5·tanh(4x) устранила проблему 
мёртвых нейронов. Новая функция активации не имеет фиксированных точек, 
обеспечивает ненулевой градиент во всём диапазоне значений, создаёт 
колебательную динамику через синусоидальный компонент. Финальная архитектура 
сочетает отбор верхних K элементов по абсолютному значению с новой функцией 
активации, слоями батч-нормализации и прореживанием с вероятностью 0.1-0.2.  Результаты показали контролируемую разреженность 75 процентов, полное 
отсутствие мёртвых нейронов (0 из 128), дисперсию 0.413 и ошибку валидации 
0.119. Увеличение ошибки на 5 процентов по сравнению с версией 3 является 
приемлемым компромиссом за устранение гибели нейронов и увеличение дисперсии. 
Систематическая проверка этого решения представлена в следующем разделе.

## 5. СИСТЕМАТИЧЕСКАЯ ПРОВЕРКА МЕТОДОМ АБЛЯЦИИ

Раздел 4 описал эволюцию архитектуры V1→V4 на основе единичных 
экспериментальных запусков. Для подтверждения надёжности результатов мы 
провели систематическую проверку методом абляции с множественными 
независимыми запусками и статистическими тестами. Каждый эксперимент 
изолирует один компонент архитектуры и измеряет его вклад в финальный 
результат.

Стратегия проверки:

- **Абляция функций активации** (раздел 5.1): необходимость sin(8x)+0.5·tanh(4x)
- **Честное сравнение с базовой архитектурой** (раздел 5.2): масштаб улучшения
- **Абляция регуляризатора дисперсии** (раздел 5.3): необходимость VarReg
- **Зависимость от параметра разреженности** (раздел 5.4): контролируемость метода
- **Обобщение на другую хаотическую систему** (раздел 5.5): переносимость
- **Стабильность обучения** (раздел 5.6): долгосрочная надёжность


### 5.1 Абляция функций активации

**Цель.** Проверить, является ли функция sin(8x)+0.5·tanh(4x) необходимой 
для достижения нулевого процента мёртвых нейронов при высокой дисперсии, 
или аналогичный результат достижим с другими функциями активации.

**Дизайн эксперимента.** Архитектура зафиксирована: latent_dim=128, K=32, 
батч-нормализация, прореживание 0.1, регуляризатор целевой дисперсии 
(λ=0.01, target=0.1). Протестированы шесть функций активации: ReLU, 
LeakyReLU(α=0.01), ELU, GELU, tanh, sin(8x)+0.5·tanh(4x). Для каждой 
проведено пять независимых запусков с полной реинициализацией весов.

**Результаты.** Таблица 1 представляет результаты абляции.

| Функция | Дисперсия | Мёртвые нейроны | Val Loss |
|---------|-----------|-----------------|----------|
| ReLU | 0.979 ± 0.082 | 45.2 ± 6.8 (35.3%) | 0.123 ± 0.002 |
| LeakyReLU | 0.027 ± 0.004 | 65.8 ± 13.5 (51.4%) | 0.114 ± 0.000 |
| ELU | 0.082 ± 0.002 | 0.0 ± 0.0 (0%) | 0.111 ± 0.000 |
| GELU | 0.037 ± 0.005 | 3.2 ± 4.4 (2.5%) | 0.114 ± 0.000 |
| tanh | 0.095 ± 0.001 | 0.0 ± 0.0 (0%) | 0.110 ± 0.000 |
| **sin8+tanh4** | **0.418 ± 0.002** | **0.0 ± 0.0 (0%)** | **0.119 ± 0.000** |

*Таблица 1: Абляция функций активации (N=5 запусков каждой). 
Жирным выделена предложенная функция.*

**Анализ.** Результаты разделяют функции активации на три категории.

*Высокая дисперсия, но мёртвые нейроны.* ReLU даёт наибольшую абсолютную 
дисперсию (0.979), однако при этом 35% нейронов мертвы. Высокая дисперсия 
является артефактом: оставшиеся активные нейроны компенсируют потерю 
мёртвых увеличением диапазона значений.

*Нулевые мёртвые нейроны, но низкая дисперсия.* ELU, GELU и tanh 
обеспечивают нулевой или близкий к нулю процент мёртвых нейронов, 
однако дисперсия не превышает 0.095.

*Парето-оптимальность.* Только sin(8x)+0.5·tanh(4x) достигает 
одновременно нулевого процента мёртвых нейронов и дисперсии 0.418. 
Среди функций с нулевым процентом мёртвых нейронов это улучшение в 
4.4 раза над вторым лучшим результатом (tanh: 0.095). Статистическая 
значимость подтверждена t-критерием Уэлча (p < 0.0001).

**Контринтуитивная находка: LeakyReLU хуже ReLU.** LeakyReLU даёт 65.8 
мёртвых нейронов против 45.2 у стандартной ReLU. Механизм объясняется 
взаимодействием с отбором по абсолютному значению: при отрицательных 
входах LeakyReLU возвращает 0.01·x, создавая малые по абсолютной 
величине значения. Механизм отбора верхних K элементов систематически 
отбрасывает эти малые значения в пользу нейронов с большими выходами, 
делая LeakyReLU-нейроны функционально мёртвыми.

![Рисунок 1: Абляция функций активации](images/activation_ablation.png)
*Рисунок 1: Сравнение шести функций активации (N=5 запусков). 
Слева: средняя дисперсия с планками погрешностей. В центре: количество 
мёртвых нейронов. Справа: ошибка валидации.*


### 5.2 Честное сравнение с базовой архитектурой

**Цель.** Исключить влияние различий в размерности латентного пространства 
и измерить чистый эффект предложенной архитектуры при равной вместимости.

**Дизайн эксперимента.** Сравниваются три архитектуры: плотная с ReLU и 
64-мерным латентным пространством (Dense_ReLU_64), плотная с ReLU и 
128-мерным пространством (Dense_ReLU_128), предложенная V4 со 128-мерным 
пространством и K=32 (V4_KSparse_128). Честное сравнение проводится между 
Dense_ReLU_128 и V4_KSparse_128, имеющими одинаковую размерность латентного 
пространства. Для каждой архитектуры проведено десять независимых запусков.

**Результаты.** Таблица 2 представляет результаты честного сравнения.

| Архитектура | Дисперсия | Мёртвые нейроны | Val Loss |
|-------------|-----------|-----------------|----------|
| Dense_ReLU_64 | 0.092 ± 0.008 | 28.5 ± 3.4 (44.5%) | 0.114 ± 0.000 |
| Dense_ReLU_128 | 0.062 ± 0.006 | 45.3 ± 6.6 (35.4%) | 0.113 ± 0.000 |
| **V4_KSparse_128** | **0.418 ± 0.002** | **0.0 ± 0.0 (0%)** | **0.120 ± 0.000** |

*Таблица 2: Честное сравнение с базовой архитектурой (N=10 запусков).*

**Честное улучшение.** При одинаковой размерности в 128 измерений V4 
обеспечивает увеличение дисперсии в 6.75 раз (0.418 / 0.062) при 
снижении доли мёртвых нейронов с 35.4% до 0%. Различие статистически 
значимо (t-критерий Уэлча, p < 0.000001).

**Парадокс вместимости.** Dense_ReLU_128 показывает меньшую дисперсию, 
чем Dense_ReLU_64 (0.062 против 0.092), несмотря на двойную вместимость. 
Причина — рост абсолютного количества мёртвых нейронов: 45.3 в 128-мерном 
пространстве против 28.5 в 64-мерном. Увеличение размерности при 
использовании ReLU усугубляет проблему, нивелируя дополнительную 
вместимость. Предложенная архитектура устраняет этот парадокс.

**Стоимость метода.** Ошибка реконструкции V4 составляет 0.120 против 
0.113 для Dense_ReLU_128, что соответствует увеличению на 6.2%. Это 
компромисс, обусловленный ограниченным диапазоном выхода функции 
sin(8x)+0.5·tanh(4x).

Воспроизводимость подтверждается исключительно низким коэффициентом 
вариации дисперсии V4: CV = 0.5% (±0.002 при среднем 0.418).

![Рисунок 2: Честное сравнение](images/fair_baseline_comparison.png)
*Рисунок 2: Честное сравнение архитектур (N=10 запусков). Верхний ряд: 
дисперсия (красная стрелка — честное сравнение 6.75×) и доля мёртвых 
нейронов. Нижний ряд: ошибка валидации и сводная таблица.*


### 5.3 Абляция регуляризатора целевой дисперсии

**Цель.** Определить, является ли регуляризатор целевой дисперсии 
(TargetVarianceRegularizer, λ=0.01, target=0.1) необходимым компонентом 
архитектуры или его можно удалить без потери качества.

**Дизайн эксперимента.** Сравниваются два условия: архитектура V4 с 
регуляризатором и без него. Остальные компоненты идентичны: K=32, 
latent_dim=128, sin(8x)+0.5·tanh(4x), батч-нормализация, прореживание. 
Для каждого условия проведено пять независимых запусков.

**Результаты.**

| Условие | Средняя дисперсия | Std |
|---------|-------------------|-----|
| С регуляризатором | 0.4179 | ± 0.0018 |
| Без регуляризатора | 0.4185 | ± 0.0011 |

Разница в средних составляет 0.0006, что соответствует 0.1% от 
абсолютного значения. T-критерий Уэлча: t = −0.54, p = 0.602. 
Различие статистически незначимо.

**Перекрёстная проверка.** Результат без регуляризатора (0.4185) 
согласуется с абляцией функций активации (раздел 5.1), где та же 
архитектура с регуляризатором дала 0.4183. Разница между 
экспериментами составляет 0.0002, подтверждая согласованность.

**Вывод.** Регуляризатор целевой дисперсии не вносит значимого вклада 
и удалён из финальной архитектуры. Высокая дисперсия является свойством 
самой функции активации sin(8x)+0.5·tanh(4x) в сочетании с механизмом 
отбора верхних K элементов, а не следствием регуляризации.

![Рисунок 3: Абляция регуляризатора](images/test_b3_varreg_comparison.png)
*Рисунок 3: Сравнение архитектуры с регуляризатором и без него (N=5 
запусков каждого условия). p = 0.602 — различие статистически незначимо.*

### 5.4 Зависимость от параметра разреженности

**Цель.** Исследовать зависимость дисперсии, мёртвых нейронов и качества 
реконструкции от параметра K и проверить монотонность масштабирования.

**Дизайн эксперимента.** Протестированы семь значений K ∈ {4, 8, 16, 32, 
64, 96, 112} при фиксированной архитектуре V4 с latent_dim=128.

**Результаты.** Таблица 3 представляет полные результаты K-абляции.

| K | Разреженность | Дисперсия | Мёртвые нейроны | Val Loss |
|---|---------------|-----------|-----------------|----------|
| 4 | 96.9% | 0.068 | 0/128 (0%) | 0.1187 |
| 8 | 93.8% | 0.131 | 0/128 (0%) | 0.1186 |
| 16 | 87.5% | 0.238 | 0/128 (0%) | 0.1187 |
| 32 | 75.0% | 0.418 | 0/128 (0%) | 0.1197 |
| 64 | 50.0% | 0.572 | 0/128 (0%) | 0.1206 |
| 96 | 25.0% | 0.651 | 0/128 (0%) | 0.1214 |
| 112 | 12.5% | 0.660 | 0/128 (0%) | 0.1218 |

*Таблица 3: Зависимость метрик от параметра K.*

**Нулевые мёртвые нейроны при всех K.** Каждая из семи конфигураций 
показала 0% мёртвых нейронов. Функция активации предотвращает гибель 
нейронов независимо от уровня разреженности, от 12.5% (K=112) до 
96.9% (K=4).

**Монотонное масштабирование.** Дисперсия монотонно возрастает с 
увеличением K: ноль нарушений тренда из шести переходов. Это 
обеспечивает предсказуемый выбор параметра K на этапе проектирования: 
для заданной целевой дисперсии существует единственное оптимальное K.

**Насыщение при высоких K.** Прирост дисперсии замедляется: переход 
K=64→96 даёт +13.8% дисперсии, а K=96→112 — лишь +1.4%. Это 
свидетельствует о том, что внутренняя размерность данных логистической 
карты 28×28 составляет порядка 96 измерений.

**Стабильность реконструкции.** Ошибка валидации варьируется от 0.1186 
до 0.1218, разброс составляет 2.7% при восьмикратном изменении 
разреженности (от 12.5% до 96.9%). Это позволяет выбирать K без 
существенного ущерба для качества реконструкции.

**Практические рекомендации.** Для задач интерпретируемости оптимален 
K=8 (разреженность 93.8%, дисперсия 0.131), что соответствует 
производственным стандартам SAE. Для задач, требующих максимальной 
дисперсии, рекомендуется K=32–64 (дисперсия 0.418–0.572). Общая 
формула выбора: K = (1 − S) × latent_dim, где S — целевая 
разреженность.

![Рисунок 4: K-абляция](images/k_sparse_ablation.png)
*Рисунок 4: Зависимость метрик от параметра K. (а) Монотонный рост 
дисперсии. (б) Соотношение разреженности и дисперсии. (в) Ноль мёртвых 
нейронов при всех K. (г) Стабильность ошибки валидации.*


### 5.5 Обобщение на карту Хенона

**Цель.** Проверить переносимость метода на хаотическую систему с 
другой динамикой.

**Дизайн эксперимента.** Архитектура V4 (K=32, latent_dim=128) обучена 
на карте Хенона (a=1.4, b=0.3) с теми же параметрами обучения. 
Результаты сопоставлены с обучением на логистической карте.

**Результаты.**

| Система | Дисперсия | Мёртвые нейроны | Val Loss |
|---------|-----------|-----------------|----------|
| Логистическая карта | 0.418 | 0/128 (0%) | 0.120 |
| Карта Хенона | 0.422 | 0/128 (0%) | 0.046 |

Соотношение дисперсий составляет 1.01 (0.422 / 0.418). Нулевой 
процент мёртвых нейронов подтверждён на обеих системах. Различие в 
ошибке валидации (0.046 против 0.120) объясняется меньшей сложностью 
данных карты Хенона, а не свойствами архитектуры.

**Вывод.** Соотношение дисперсий, близкое к единице, свидетельствует 
о том, что предложенный метод отражает общие свойства хаотических 
аттракторов и не подстраивается под конкретную динамику.

![Рисунок 5: Обобщение на карту Хенона](images/henon_generalization.png)
*Рисунок 5: Сравнение логистической карты и карты Хенона. Верхний ряд: 
проекции латентного пространства. Нижний ряд: распределение дисперсий 
(соотношение 1.01).*


### 5.6 Стабильность обучения

**Цель.** Подтвердить, что нулевой процент мёртвых нейронов сохраняется 
на протяжении всего процесса обучения, а не является свойством только 
конечного состояния модели.

**Дизайн эксперимента.** Отслеживание количества мёртвых нейронов и 
средней дисперсии после каждой эпохи в течение 30 эпох обучения 
архитектуры V4 (K=32, latent_dim=128).

**Результаты.** Количество мёртвых нейронов оставалось на нулевом 
уровне во все 30 эпох. Дисперсия стабилизировалась на уровне 
0.417 ± 0.002 начиная с четвёртой эпохи, демонстрируя минимальные 
колебания (коэффициент вариации менее 1%).

Дополнительная статистическая проверка (N=5 независимых запусков) 
подтвердила воспроизводимость: дисперсия 0.419 ± 0.001, мёртвые 
нейроны 0.0 ± 0.0 во всех запусках.

**Вывод.** Свойство нулевой гибели нейронов является стабильным на 
протяжении всего обучения и не деградирует с увеличением числа эпох. 
Это является важной гарантией для приложений, требующих длительного 
обучения.

![Рисунок 6: Стабильность обучения](images/training_stability.png)
*Рисунок 6: Стабильность обучения V4 в течение 30 эпох. Слева: 
мёртвые нейроны остаются на нуле. Справа: дисперсия стабилизируется 
с минимальными колебаниями (±0.002).*

### 5.7 Временная динамика: известное ограничение

Мы проверили, сохраняют ли скрытые представления экспоненциальную дивергенцию, характерную для хаотических систем, путем кодирования двух траекторий с начальными условиями, отличающимися на δx₀ = 10⁻⁶.

**Результаты:**

```
Исходный уровень плотного ReLU (latent_dim=64):
  Расстояния: 3,74 ± 0,37
  Соотношение (конечное/начальное): 1,30
  Дисперсия: 0,085
  Количество погибших нейронов: 30/64 (46,9%)

V4 K-Разреженный хаос (K=32, latent_dim=128):
  Расстояния: 9,28 ± 0,76
  Соотношение (конечное/начальное): 1,07
  Дисперсия: 0,418
  Количество погибших нейронов: 0/128 (0%)

Ожидаемое сохранение хаоса: коэффициент >5,0
```

**Вывод:** Ни одна из архитектур прямой связи не сохраняет экспоненциальную дивергенцию (оба соотношения ~1,0-1,3 против ожидаемого >5,0). Однако наш метод обеспечивает значительно более высокое разделение траекторий (2,5×), дисперсию (4,9×) и полностью исключает гибель нейронов (0% против 46,9%).

**Интерпретация:** Наша архитектура прямой трансляции сохраняет ** свойства статического хаоса ** (высокая дисперсия, разнообразие функций), но не ** временную динамику ** (экспоненциальное расхождение). Это ожидаемое ограничение: автоэнкодеры прямой трансляции обрабатывают каждый кадр независимо без использования временной памяти.

**Сфера применения:** Наш метод подходит для приложений, основанных на энтропии (криптография, статическая интерпретируемость), где важна дисперсия, но не для прогнозирования временных рядов или оценки показателя Ляпунова, для которых требуются рекуррентные архитектуры (LSTM, резервуарные вычисления [4]). Исследование устойчивости хаотических систем из данных [5] и сохранения динамических свойств в латентных пространствах автоэнкодеров [6] представляют смежные направления, однако ни одна из этих работ не рассматривает одновременное обеспечение разреженности.



**Рисунок 6:** Сравнение хаотической дивергенции. (а) Истинная логистическая карта: экспоненциальное разделение (λ ≈ 0,5). (б) Плотный ReLU: стабильно низкие расстояния (~3-4), высокая гибель нейронов. (c) V4 K-Разреженный хаос: стабильно большие расстояния (~9), нулевая гибель нейронов. Ни то, ни другое не сохраняет временной хаос, но V4 сохраняет превосходные статические свойства.



### 5.8 Абляция коэффициентов функции активации

**Цель.** Определить чувствительность дисперсии и мёртвых нейронов к каждому 
из трёх коэффициентов функции sin(ax) + w·tanh(bx) и найти направления 
оптимизации.

**Дизайн эксперимента.** Каждый коэффициент варьируется независимо при 
фиксации двух остальных на значениях по умолчанию (a=8, w=0.5, b=4). 
Протестированы: a ∈ {2, 4, 6, 8, 10, 12, 16}, w ∈ {0.0, 0.1, 0.25, 0.5, 
0.75, 1.0}, b ∈ {1, 2, 4, 8, 12, 16}. Для каждой конфигурации проведено 
N=10 независимых запусков. Логистическая карта, 2000 сэмплов, 10 эпох.

**Результаты.** Таблица N представляет результаты абляции коэффициентов.

| Параметр | Значение | Дисперсия | Мёртвые | Val Loss |
|----------|----------|-----------|---------|----------|
| a (частота sin) | 4 | 0.4592 ± 0.0014 | 0 | 0.1181 |
| | 8 (контроль) | 0.4187 ± 0.0019 | 0 | 0.1180 |
| | 16 | 0.4132 ± 0.0029 | 0 | 0.1180 |
| w (вес tanh) | 0.0 (только sin) | 0.2362 ± 0.0002 | 0 | 0.1178 |
| | 0.5 (контроль) | 0.4187 ± 0.0025 | 0 | 0.1180 |
| | 1.0 | 0.6914 ± 0.0058 | 0 | 0.1187 |
| b (крутизна tanh) | 1 | 0.3315 ± 0.0010 | 0 | 0.1179 |
| | 4 (контроль) | 0.4178 ± 0.0019 | 0 | 0.1181 |
| | 16 | 0.4689 ± 0.0041 | 0 | 0.1181 |

**Нулевые мёртвые нейроны при всех значениях.** Все 19 конфигураций 
(включая w=0, чистый синус без tanh) показали 0/128 мёртвых нейронов. 
Свойство нулевой гибели определяется наличием синусоидального компонента 
и не зависит от конкретных значений коэффициентов.

**Вес tanh — доминирующий фактор.** Параметр w оказывает наибольшее 
влияние на дисперсию: увеличение от 0.0 до 1.0 даёт рост дисперсии в 
2.93 раза (от 0.236 до 0.691). Для сравнения, оптимальная частота a=4 
даёт прирост лишь в 1.10 раза относительно контроля, а крутизна b=16 — 
в 1.12 раза. При этом потеря качества реконструкции при w=1.0 составляет 
менее 0.6%.

**Оптимальная комбинация.** На основании одномерной абляции определена 
кандидатная конфигурация: a=4, w=1.0, b=16. Валидация данной комбинации 
на множественных доменах представлена в разделе 6.5.

![Рисунок: Абляция коэффициентов](images/coefficient_ablation.png)
![Рисунок: Абляция коэффициентов (линии)](images/coefficient_ablation_lines.png)

## 6. МАСШТАБИРОВАНИЕ И ГРАНИЧНОЕ ТЕСТИРОВАНИЕ

### 6.1 Цель раздела

Разделы 1–5 проводили все эксперименты в контролируемых условиях: две 
хаотические системы с внутренней размерностью порядка 1–2 (логистическая 
карта и карта Хенона), обучающая выборка размером 2000 сэмплов, 
латентное пространство размерности 128. Эти условия обеспечили чистоту 
абляционного анализа, однако оставляют открытым вопрос: какие из 
обнаруженных свойств архитектуры V4 являются общими (архитектурными), 
а какие зависят от конкретного домена данных?

Данный раздел систематически проверяет три аспекта обобщаемости:

- **Масштабирование по объёму данных** (раздел 6.2): исключает артефакт 
  малого датасета путём увеличения обучающей выборки до 20000 сэмплов.
- **Высокоразмерный хаос** (раздел 6.3): проверяет метод на системе 
  Лоренца-96 с внутренней размерностью порядка 13–15, что на порядок 
  выше исходных систем.
- **Граничное тестирование на структурированных данных** (раздел 6.4): 
  применяет метод к MNIST для разделения общих и доменно-специфичных 
  свойств.

### 6.2 Масштабирование по объёму данных

**Цель.** Исключить возможность того, что нулевой процент мёртвых нейронов 
и стабильная дисперсия ~0.41 являются артефактами малой обучающей выборки 
(2000 сэмплов).

**Дизайн эксперимента.** Логистическая карта при четырёх размерах обучающей 
выборки: 2000, 5000, 10000, 20000 сэмплов. Сравниваются V4_KSparse_Chaos 
(latent_dim=128, K=32) и Dense_ReLU_128. Для каждой конфигурации проведено 
N=3 независимых запуска. Обучение: 10 эпох, batch size 64, Adam(lr=0.001).

**Результаты.** Таблица 5 представляет результаты масштабирования.

| Размер | V4 дисперсия | V4 мёртвые | Dense дисперсия | Dense мёртвые | Ratio | p-value |
|--------|-------------|-----------|----------------|-------------|-------|---------|
| 2K | 0.4183 ± 0.0005 | 0 (0%) | 0.2406 ± 0.0049 | 0 (0%) | 1.74× | <0.000001 |
| 5K | 0.4131 ± 0.0011 | 0 (0%) | 0.3417 ± 0.1052 | 0 (0%) | 1.21× | 0.391 |
| 10K | 0.4077 ± 0.0018 | 0 (0%) | 0.5726 ± 0.2419 | 0 (0%) | 0.71× | 0.390 |
| 20K | 0.4040 ± 0.0000 | 0 (0%) | 1.2936 ± 0.2373 | 0 (0%) | 0.31× | 0.006 |

*Таблица 5: Масштабирование по объёму данных (N=3 запуска). Ratio — 
отношение дисперсии V4 к Dense_ReLU.*

**Стабильность V4.** Средняя дисперсия V4 варьируется в узком диапазоне 
от 0.4040 до 0.4183 при увеличении датасета в 10 раз. Стандартное 
отклонение средних по четырём размерам составляет 0.0054. Мёртвые 
нейроны остаются на нуле во всех 12 запусках (4 размера × 3 запуска). 
Незначительное монотонное снижение дисперсии с ростом размера выборки 
(−3.4% от 2K к 20K) указывает на лёгкую стабилизацию латентных 
представлений при увеличении объёма данных, не затрагивающую свойство 
нулевой гибели нейронов.

**Нестабильность Dense_ReLU.** Дисперсия Dense_ReLU демонстрирует 
пятикратный рост от 0.24 при 2000 сэмплах до 1.29 при 20000. 
Стандартное отклонение внутри запусков увеличивается с 0.005 (при 2K) 
до 0.237 (при 20K), что свидетельствует о нарастающей зависимости 
результатов от инициализации весов. При 20000 сэмплах отдельные запуски 
Dense_ReLU дают дисперсию от 0.997 до 1.578 — разброс в 1.6 раза при 
фиксированных гиперпараметрах.

**Вывод.** Свойства V4 не являются артефактом малого датасета. 
Стабильность дисперсии (CV = 1.3% по четырём размерам) и нулевой 
процент мёртвых нейронов сохраняются при десятикратном увеличении 
обучающей выборки. Напротив, Dense_ReLU становится всё менее 
предсказуемой с ростом объёма данных.

![Рисунок 7: Масштабирование по объёму данных](images/dataset_scaling.png)
*Рисунок 7: Масштабирование по объёму данных (N=3 запуска). Слева: 
дисперсия V4 стабильна (синяя полоса), Dense_ReLU нестабильна 
(оранжевая полоса с растущим разбросом). Справа: нулевые мёртвые 
нейроны V4 на всех размерах.*


### 6.3 Высокоразмерный хаос: Система Лоренца-96

**Цель.** Проверить обобщаемость метода на хаотической системе с 
внутренней размерностью порядка 13–15, что на порядок выше логистической 
карты (dim ~1) и карты Хенона (dim ~2).

**Система Лоренца-96.** Модель описывается уравнениями 
dx_i/dt = (x_{i+1} − x_{i−2})·x_{i−1} − x_i + F для i = 1, …, 40 
с параметром форсирования F = 8.0. При данном значении F система 
демонстрирует хаотическое поведение с положительными показателями 
Ляпунова и внутренней размерностью порядка 13–15 на аттракторе. 
Изображения 28×28 генерируются из первой переменной состояния x_1.

**Дизайн эксперимента.** Три архитектуры: V4_KSparse_Chaos, TopK_ReLU, 
Dense_ReLU (все с latent_dim=128). V4 и TopK используют K=32 (75% 
разреженность). Обучение на двух системах: Лоренц-96 (5000 сэмплов) и 
логистическая карта (5000 сэмплов, для перекрёстного сравнения). 
N=5 независимых запусков, 10 эпох, batch size 64.

**Результаты.** Таблица 6 представляет результаты по двум системам.

| Система | Архитектура | Дисперсия | Мёртвые нейроны | Val Loss |
|---------|-------------|-----------|-----------------|----------|
| Лоренц-96 | V4_KSparse_Chaos | 0.4124 ± 0.0018 | 0.0 ± 0.0 (0%) | 0.0432 ± 0.0000 |
| Лоренц-96 | TopK_ReLU | 1.8087 ± 0.0935 | 12.6 ± 1.2 (9.8%) | 0.0323 ± 0.0003 |
| Лоренц-96 | Dense_ReLU | 0.5709 ± 0.0292 | 0.0 ± 0.0 (0%) | 0.0286 ± 0.0014 |
| Логистич. | V4_KSparse_Chaos | 0.4133 ± 0.0031 | 0.0 ± 0.0 (0%) | 0.1166 ± 0.0001 |
| Логистич. | TopK_ReLU | 1.1074 ± 0.0990 | 13.2 ± 4.1 (10.3%) | 0.1126 ± 0.0003 |
| Логистич. | Dense_ReLU | 0.3857 ± 0.0915 | 0.0 ± 0.0 (0%) | 0.1198 ± 0.0017 |

*Таблица 6: Лоренц-96 vs Логистическая карта (N=5 запусков). Три 
архитектуры на двух хаотических системах.*

**Перекрёстная стабильность V4.** Соотношение дисперсий V4 между 
системами составляет 0.4124 / 0.4133 = 1.00× (коэффициент обобщения). 
Переход от внутренней размерности ~1 к ~13–15 не повлиял на дисперсию 
латентного пространства. Нулевой процент мёртвых нейронов сохраняется 
на обеих системах во всех 10 запусках.

**Систематическая деградация TopK_ReLU.** На Лоренце-96 TopK_ReLU теряет 
12.6 нейронов (9.8% от 128), на логистической карте — 13.2 нейрона (10.3%). 
Проблема мёртвых нейронов при использовании ReLU не специфична для 
одномерного хаоса: она воспроизводится на системе с принципиально иной 
динамикой и размерностью. Дисперсия TopK_ReLU нестабильна между системами 
(1.81 vs 1.11, соотношение 1.63×), что указывает на зависимость от 
статистических свойств входных данных.

**Статистическая значимость.** Различие дисперсий V4 и TopK_ReLU на 
Лоренце-96 высоко значимо (t-критерий Уэлча, p < 0.00001). Различие 
дисперсий V4 и Dense_ReLU также значимо (p < 0.0004).

**Вывод.** Метод обобщается на хаотическую систему с нетривиальной 
внутренней размерностью. Свойства нулевой гибели нейронов и стабильной 
дисперсии ~0.41 не зависят от конкретной динамической системы и её 
размерности. TopK_ReLU систематически теряет ~10% нейронов независимо 
от домена.

![Рисунок 8: Лоренц-96](images/lorenz96_test.png)
*Рисунок 8: Обобщение на систему Лоренца-96 (N=5 запусков). Сравнение 
трёх архитектур на двух хаотических системах. V4 демонстрирует идеальную 
перекрёстную стабильность (коэффициент 1.00×).*


### 6.4 Граничное тестирование: структурированные данные (MNIST)

**Цель.** Определить границы применимости метода путём тестирования на 
реальных структурированных данных, принципиально отличающихся от 
хаотических систем. MNIST представляет домен низкой энтропии со 
структурированными паттернами (рукописные цифры), в отличие от 
высокоэнтропийных хаотических данных разделов 1–7.

**Дизайн эксперимента.** MNIST: 60000 обучающих, 10000 тестовых 
изображений 28×28. Три архитектуры: V4_KSparse_Chaos, TopK_ReLU, 
Dense_ReLU (latent_dim=128). V4 и TopK используют K=32 (75% 
разреженность). N=5 независимых запусков, 10 эпох, batch size 128.

**Результаты.** Таблица 7 представляет результаты на MNIST.

| Архитектура | Дисперсия | Мёртвые нейроны | Разреженность | Val Loss (MSE) |
|-------------|-----------|-----------------|---------------|----------------|
| V4_KSparse_Chaos | 0.4097 ± 0.0033 | 0.0 ± 0.0 (0%) | 75.0% | 0.0671 ± 0.0001 |
| TopK_ReLU | 3511.3 ± 549.6 | 61.0 ± 5.7 (47.7%) | 75.0% | 0.0113 ± 0.0004 |
| Dense_ReLU | 1 632 276 ± 496 001 | 0.0 ± 0.0 (0%) | 0.0% | 0.1207 ± 0.0022 |

*Таблица 7: Тестирование на MNIST (N=5 запусков). Dense_ReLU без 
механизма разреживания указан для полноты сравнения.*

**Подтверждённое общее свойство: нулевые мёртвые нейроны.** V4 
сохраняет нулевой процент мёртвых нейронов на реальных 
структурированных данных. Это свойство подтверждено на четвёртом 
домене данных (после логистической карты, карты Хенона и Лоренца-96) 
и сохраняется при тридцатикратном увеличении объёма обучающей выборки 
(60000 vs 2000).

**Подтверждённое общее свойство: стабильная дисперсия.** Дисперсия V4 
на MNIST составляет 0.4097 ± 0.0033, что практически идентично 
результатам на хаотических системах (0.413 на логистической карте, 
0.412 на Лоренце-96). Соотношение дисперсий MNIST / Логистическая карта 
= 0.99×. Дисперсия архитектуры V4 определяется свойствами функции 
активации sin(8x)+0.5·tanh(4x), а не характеристиками входных данных.

**Усиление проблемы мёртвых нейронов TopK_ReLU.** На MNIST TopK_ReLU 
теряет 61 нейрон (47.7%), что значительно хуже результатов на хаотических 
системах (12.6 на Лоренце-96, то есть 9.8%). Проблема мёртвых нейронов 
при использовании ReLU с механизмом Top-K не только сохраняется, но 
усиливается на структурированных данных. Статистическая значимость 
различия V4 и TopK_ReLU по дисперсии: p < 0.0003.

**Выявленное доменно-специфичное свойство: качество реконструкции.** 
Ошибка реконструкции V4 на MNIST (MSE 0.0671) существенно хуже, чем 
у TopK_ReLU (0.0113). TopK_ReLU обеспечивает в 5.9 раза лучшую 
реконструкцию на структурированных данных. Визуальный анализ 
реконструкций подтверждает это: V4 производит размытые контуры цифр 
с потерей мелких деталей, тогда как TopK_ReLU восстанавливает чёткие, 
узнаваемые цифры.

При этом V4 показывает значительно лучшую реконструкцию, чем Dense_ReLU 
(0.0671 vs 0.1207), что свидетельствует о том, что комбинация механизма 
Top-K с хаотической активацией сохраняет значимую реконструктивную 
способность даже на неоптимальном для неё домене.

![Рисунок 9: MNIST тест](images/mnist_test.png)
*Рисунок 9: Тестирование на MNIST (N=5 запусков). V4 сохраняет нулевые 
мёртвые нейроны и стабильную дисперсию, но уступает TopK_ReLU в 
качестве реконструкции.*

![Рисунок 10: Реконструкции MNIST](images/mnist_reconstruction.png)
*Рисунок 10: Визуальное сравнение реконструкций MNIST. Верхний ряд: 
оригиналы. V4: размытые формы с сохранением общей структуры. 
TopK_ReLU: чёткие узнаваемые цифры.*


### 6.5 Валидация оптимальных коэффициентов на всех доменах

**Цель.** Проверить, переносится ли комбинация коэффициентов (a=4, w=1.0, 
b=16), найденная при одномерной абляции на логистической карте (раздел 5.8), 
на все домены из разделов 6.2–6.4.

**Дизайн эксперимента.** Сравниваются три конфигурации: Original (8, 0.5, 4), 
Optimized (4, 1.0, 16) и TopK_ReLU (baseline). Домены: логистическая карта 
(5K), система Лоренца-96 (5K), MNIST (60K). Для каждой конфигурации N=10 
независимых запусков. Дополнительно проведено масштабирование по объёму данных 
(2K–20K) на логистической карте.

**Результаты.** Таблица M представляет результаты мультидоменного сравнения.

| Домен | Original var | Optimized var | Ratio | Dead (O/Opt) | Loss gap |
|-------|-------------|--------------|-------|-------------|----------|
| Logistic | 0.413 ± 0.003 | 0.784 ± 0.009 | 1.90× | 0/0 | +0.6% |
| Lorenz-96 | 0.412 ± 0.003 | 0.772 ± 0.011 | 1.87× | 0/0 | +1.0% |
| MNIST | 0.410 ± 0.003 | 0.793 ± 0.013 | 1.94× | 0/0 | −3.9% |

Все различия статистически значимы (p < 10⁻⁶, Welch t-test).

**Стабильность при масштабировании.** На логистической карте преимущество 
Optimized сохраняется при всех размерах выборки: от 1.92× при 2K до 1.84× 
при 20K (все p < 10⁻⁶). Мёртвые нейроны остаются на нуле во всех 80 запусках 
(4 размера × 2 конфигурации × 10 запусков). Незначительное снижение ratio с 
ростом выборки (−4%) указывает на стабилизацию латентных представлений при 
увеличении объёма данных.

**Улучшение реконструкции на MNIST.** На структурированных данных Optimized 
не только увеличивает дисперсию, но и улучшает качество реконструкции на 3.9% 
(loss 0.0641 vs 0.0667). Это свидетельствует о том, что более пологий 
синусоидальный компонент (a=4 vs a=8) и усиленный стабилизирующий компонент 
(w=1.0, b=16) обеспечивают более эффективное кодирование на данных с 
выраженной структурой.

**Компромисс на хаотических данных.** На логистической карте и Лоренце-96 
увеличение дисперсии сопровождается незначительным ростом ошибки реконструкции 
(+0.6% и +1.0% соответственно). Данный компромисс может быть приемлем для 
приложений, где высокая дисперсия латентного пространства является приоритетом.

![Рисунок: Мультидоменное сравнение коэффициентов](images/coefficient_combination_multidomain.png)
![Рисунок: Масштабирование коэффициентов](images/coefficient_combination_scaling.png)


### 6.6 Анализ: разделение общих и доменно-специфичных свойств

Совокупность результатов разделов 6.2–6.4 позволяет классифицировать 
свойства архитектуры V4 по типу обобщаемости.

**Таблица 8: Классификация свойств.**

| Свойство | Тип | Механизм |
|----------|-----|----------|
| Нулевые мёртвые нейроны | Общее | Высокая производная в нуле (~10), отсутствие «мёртвой зоны» |
| Стабильная дисперсия ~0.41 | Общее | Ограниченный выход [-1.5, 1.5], симметрия функции |
| Качественная реконструкция на хаосе | Доменно-специфичное | sin(8x) рассеивает значения — полезно для высокоэнтропийных данных |
| Ухудшенная реконструкция на MNIST | Доменно-специфичное | sin(8x) «хаотифицирует» структуру — вредно для низкоэнтропийных данных |

**Общие свойства (подтверждены на 4 доменах).** Нулевой процент мёртвых 
нейронов воспроизведён на логистической карте (2K–20K сэмплов), карте 
Хенона, Лоренце-96 (dim=40) и MNIST (60K сэмплов). Механизм является 
следствием отсутствия фиксированных точек у функции sin(8x)+0.5·tanh(4x): 
для любого входного значения x выполняется f(x) ≠ x, что означает 
невозможность «застревания» нейрона в состоянии, из которого отбор Top-K 
его систематически исключает. Высокая производная функции в нуле (~10 
против 1 у ReLU) обеспечивает устойчивый градиентный поток даже для 
нейронов с малыми пре-активациями.

Стабильная дисперсия ~0.41 также является общим свойством: значение 
определяется ограниченным диапазоном функции активации [-1.5, 1.5] и 
фиксированными архитектурными параметрами (K=32, latent_dim=128), а не 
статистическими свойствами входных данных. Коэффициент вариации дисперсии 
по четырём доменам составляет менее 1%.

**Доменно-специфичные свойства.** Качество реконструкции зависит от 
совместимости осциллирующей компоненты sin(8x) со статистической 
структурой данных. На высокоэнтропийных данных (хаотические системы) 
периодическое рассеивание значений полезно: оно обеспечивает равномерное 
покрытие пространства активаций и предотвращает коллапс распределения. 
На низкоэнтропийных данных (MNIST) тот же механизм вреден: близкие 
входные значения, соответствующие семантически схожим пикселям, 
отображаются в разнесённые точки пространства активаций, разрушая 
пространственную когерентность представления.

**Ключевой вывод.** Принцип активаций без фиксированных точек для 
устранения мёртвых нейронов является общим и доменно-независимым. 
Конкретная реализация через sin(8x)+0.5·tanh(4x) оптимизирована под 
высокоэнтропийный домен. Для структурированных данных необходима 
функция, сохраняющая свойство отсутствия фиксированных точек, но 
заменяющая осциллирующий компонент на монотонный.


### 6.7 Обсуждение и направления развития

**Доказанный универсальный принцип.** Тестирование на четырёх доменах 
различной природы и размерности подтверждает: активации без фиксированных 
точек (f(x) ≠ x для всех x в рабочей области) в сочетании с механизмом 
Top-K устраняют проблему мёртвых нейронов. Это не свойство конкретной 
формулы, а следствие топологического ограничения на функцию активации.

**Первая реализация, оптимальная для высокоэнтропийного домена.** Формула 
sin(8x)+0.5·tanh(4x) является первой конкретной реализацией данного 
принципа, продемонстрировавшей парето-оптимальность по критериям дисперсии 
и гибели нейронов. Её область оптимальности — данные с высокой энтропией 
(хаотические системы, генерация случайности, криптографические приложения).

**Направления расширения.**

*Монотонные активации без фиксированных точек для структурированных данных.* 
Результаты на MNIST (раздел 6.4) указывают на необходимость функции 
активации, сочетающей свойство f(x) ≠ x с монотонностью. Кандидаты 
включают сдвинутые ограниченные функции вида f(x) = tanh(x) + ε для 
малого ε > 0, которые сохраняют пространственную когерентность при 
устранении фиксированной точки. Систематическая абляция по типу монотонной 
деформации и величине смещения является непосредственным следующим шагом.

*Обучаемые активации с ограничением на отсутствие фиксированных точек.* 
Параметризация функции активации с обучаемыми коэффициентами при жёстком 
ограничении f(x) ≠ x позволит автоматически адаптировать форму активации 
под домен данных. Это потребует разработки дифференцируемой штрафной 
функции, обеспечивающей отсутствие фиксированных точек на рабочем 
интервале.

*Применение принципа к архитектурам уровня SOTA.* Современные методы 
разреженных автоэнкодеров — Gated SAE [7] (NeurIPS 2024), 
JumpReLU [8] (ICLR 2025) — используют функции активации с фиксированными точками (ReLU: 
f(0) = 0, Heaviside step). Интеграция принципа fixed-point-free активаций 
в эти архитектуры может устранить проблему мёртвых нейронов, 
документированную в соответствующих работах, без радикального 
изменения общей структуры.


## 7. ОБСУЖДЕНИЕ

Систематическая проверка методом абляции подтвердила необходимость каждого 
компонента предложенной архитектуры. Данный раздел анализирует механизмы 
работы метода, границы его применимости и обнаруженные ограничения.

### 7.1 Аномалия LeakyReLU: взаимодействие с отбором по абсолютному значению

Абляция функций активации в разделе 5.1 выявила контринтуитивный результат. 
Функция LeakyReLU с коэффициентом 0.01 дала 66 мёртвых нейронов из 128 что 
на 47 процентов хуже чем стандартная ReLU с 45 мёртвыми нейронами. Это 
противоречит общепринятому представлению что LeakyReLU предотвращает гибель 
нейронов благодаря ненулевому градиенту для отрицательных значений.

Механизм данного эффекта связан с взаимодействием LeakyReLU и механизма 
отбора верхних K элементов по абсолютному значению. Для отрицательных 
входов функция LeakyReLU даёт выход 0.01x где x — входное значение. При 
отборе верхних K элементов сравниваются абсолютные величины всех активаций. 
Значение abs(0.01x) в сто раз меньше abs(x) для положительных активаций 
того же масштаба. В результате отрицательные пре-активации после LeakyReLU 
практически никогда не попадают в верхние 32 элемента из 128 и систематически 
обнуляются механизмом разреживания делая эти измерения функционально мёртвыми 
независимо от наличия градиента.

Данное наблюдение имеет практическое значение для проектирования разреженных 
архитектур. Любая функция активации с сильно асимметричным наклоном где 
коэффициент для отрицательных значений значительно меньше единицы будет 
страдать от аналогичного эффекта при использовании отбора по абсолютному 
значению. Ограниченные функции активации с почти симметричным диапазоном 
выхода такие как гиперболический тангенс, ELU и предложенная sin(8x)+0.5·tanh(4x) 
избегают этой проблемы. Аномалия LeakyReLU служит предостережением против 
слепого применения стандартных решений без учёта специфики механизма 
разреживания.

### 7.2 Механизм работы функции активации sin(8x)+0.5·tanh(4x)

Предложенная функция активации обеспечивает увеличение дисперсии в 4.4 раза 
по сравнению с гиперболическим тангенсом при идентичной архитектуре согласно 
результатам абляции в разделе 5.1. Механизм данного эффекта связан с 
рассеиванием значений синусоидальным компонентом. Период функции синус равен 
2π делённое на 8 что составляет приблизительно 0.785 радиан. В типичном 
диапазоне пре-активаций от минус 3 до плюс 3 укладывается около 8 полных 
периодов синусоиды. Различные входные значения равномерно распределяются по 
всему диапазону выхода от минус 1 до плюс 1 создавая широкое распределение 
активаций.

Гиперболический тангенс напротив демонстрирует эффект насыщения. При 
абсолютных значениях входа больше 2 выход tanh(x) практически достигает 
границ плюс-минус 1 независимо от точного значения входа. Большинство 
пре-активаций сжимается к узкому диапазону вблизи границ что приводит к 
низкой дисперсии выходов. Компонент 0.5·tanh(4x) в предложенной функции 
добавляет монотонное смещение которое стабилизирует обучение и обеспечивает 
ненулевой градиент во всём диапазоне значений предотвращая проблему 
исчезающего градиента.

Ограниченный выход функции в диапазоне от минус 1.5 до плюс 1.5 предотвращает 
гибель нейронов под действием механизма отбора по абсолютному значению так 
как все активации остаются сопоставимыми по масштабу. Однако периодичность 
синусоидальной компоненты приводит к потере информации. Несколько различных 
входных значений отстоящих друг от друга на период синуса отображаются в 
практически одинаковый выход. Это объясняет увеличение ошибки реконструкции 
на 8.2 процента по сравнению с гиперболическим тангенсом.

**Роль отдельных коэффициентов.** Систематическая абляция 19 конфигураций 
коэффициентов (раздел 5.8) выявила иерархию влияния. Вес компонента tanh (w) 
является доминирующим фактором: увеличение w от 0.0 до 1.0 даёт рост 
дисперсии в 2.93 раза при потере качества реконструкции менее 0.6%. Частота 
синуса (a) и крутизна tanh (b) оказывают значительно меньшее влияние — 
прирост дисперсии не превышает 1.12 раза относительно контрольных значений.

Критически важно, что свойство нулевой гибели нейронов сохраняется при всех 
19 конфигурациях, включая случай w=0 (чистый синус без tanh). Это 
подтверждает, что нулевая гибель является следствием наличия осциллирующего 
компонента как такового, а не конкретного соотношения коэффициентов.

На основании одномерной абляции определена оптимизированная комбинация 
(a=4, w=1.0, b=16), обеспечивающая увеличение дисперсии в 1.9 раза 
относительно исходной формулы. Мультидоменная валидация (раздел 6.5) 
подтвердила стабильность этого преимущества на всех трёх доменах 
(логистическая карта, Лоренц-96, MNIST) при сохранении нулевого процента 
мёртвых нейронов. На MNIST оптимизированная комбинация дополнительно 
улучшает реконструкцию на 3.9%, что указывает на то, что более пологий 
синусоидальный компонент (a=4 vs a=8) лучше сохраняет пространственную 
когерентность структурированных данных.

Открытым остаётся вопрос совместного влияния коэффициентов: проведённая 
абляция варьировала каждый параметр независимо, и взаимодействие между 
коэффициентами не исследовалось. Полный факторный эксперимент по сетке 
коэффициентов является направлением для будущей работы.

### 7.3 Обобщение на другие хаотические системы

Результаты раздела 5.5 показали что архитектура обобщается на карту Хенона 
с соотношением дисперсий 1.01. Это указывает что метод отражает общие 
свойства хаотических аттракторов а не подстраивается под конкретную 
динамическую систему. Полное сохранение нулевого процента мёртвых нейронов 
и практически идентичная дисперсия подтверждают что архитектурные свойства 
переносятся без переобучения на специфику логистической карты.

Однако обе протестированные системы имеют очень низкую внутреннюю размерность 
порядка 1-2 так как хаотические траектории полностью параметризованы 
небольшим числом начальных условий. Внутренняя размерность данных 
логистической карты фактически равна 1 так как каждое изображение 
генерируется из единственного начального значения x0. Использование 32 
активных нейронов из 128 для данных с внутренней размерностью 1 является 
переполнением в 32 раза.

Тестирование на системе Лоренца-96 с внутренней размерностью порядка 13-15 
(раздел 6.3) и на MNIST (раздел 6.4) подтвердило обобщаемость метода за 
пределы низкоразмерных хаотических карт. Свойство нулевой гибели нейронов 
и стабильной дисперсии ~0.41 сохраняется на всех четырёх доменах. Однако 
масштабирование на данные с ещё более высокой внутренней размерностью, 
такие как CIFAR с размерностью порядка 50 или векторные представления 
токенов в больших языковых моделях с размерностью порядка 100 и более, 
остаётся открытым вопросом.

### 7.4 Практическая применимость

Обнаруженные свойства архитектуры имеют прямую практическую применимость 
в нескольких областях машинного обучения и развёртывания моделей.

**Развёртывание на периферийных устройствах.** Разреженность 75 процентов 
означает что только 32 из 128 элементов латентного вектора активны после 
каждого прохода. Плотное латентное пространство размерности 128 требует 
512 байт для хранения в формате float32. Разреженное представление с 32 
активными элементами требует 128 байт для значений и 32 байта для индексов 
в формате uint8 при условии что размерность латентного пространства не 
превышает 256. Итоговая вместимость составляет 160 байт что даёт сжатие в 
3.2 раза. При использовании uint16 для индексов при размерности больше 256 
требуется 64 байта для индексов что даёт итоговую вместимость 192 байта и 
сжатие в 2.67 раза.

Теоретический максимум ускорения вывода при разреженности 75 процентов 
составляет 4 раза так как активны только 25 процентов элементов. Однако 
практическое ускорение составляет от 2 до 3 раз с учётом накладных расходов 
на операцию отбора верхних K элементов сложностью O(n log k) и операции 
scatter-gather для работы с разреженными индексами. Точная оценка требует 
бенчмарка на целевом оборудовании так как ускорение сильно зависит от 
аппаратной платформы и наличия специализированных инструкций для разреженных 
операций.

**Системы долгосрочного обучения.** Тестирование показало сохранение нулевых 
мёртвых нейронов в течение 30 эпох обучения при множественных запусках что 
критично для приложений в робототехнике, обучении с подкреплением и 
производственных системах машинного обучения. Накопление мёртвых нейронов 
со временем приводит к деградации производительности модели и необходимости 
периодического переобучения. Гарантия стабильности обеспечиваемая нулевым 
процентом мёртвых нейронов на протяжении всего обучения устраняет эту 
проблему.

**Представления с сохранением приватности.** Разреженность 75 процентов 
означает что передаётся только четверть латентного пространства снижая объём 
потенциально утекающей информации. Ограниченный диапазон выхода функции 
активации от минус 1.5 до плюс 1.5 исключает выбросы больших значений которые 
могут непреднамеренно кодировать чувствительную информацию. Высокая дисперсия 
0.418 в активных 25 процентах элементов создаёт зашумлённое представление 
которое труднее инвертировать обратно к исходным данным что важно для 
приложений в медицинской визуализации, федеративном обучении и обработке 
финансовых данных.

**Предсказуемый контроль разреженности.** Монотонная зависимость дисперсии 
от K позволяет интерполировать требуемое значение параметра под ограничения 
конкретной задачи без подбора гиперпараметров методом проб и ошибок. Например 
если система имеет бюджет задержки 10 миллисекунд и текущая архитектура с K 
равным 32 занимает 8 миллисекунд то максимально допустимое K равно 32 
умножить на 10 делённое на 8 что даёт 40. Ожидаемая дисперсия при K равном 
40 может быть предсказана интерполяцией между значениями для K равных 32 и 
64 без необходимости переобучения модели.

### 7.5 Ограничения

**Исходная гипотеза не подтвердилась: temporal chaos NOT preserved.** 
Для проверки сохранения хаотической динамики в латентном пространстве 
две траектории логистической карты с близкими начальными условиями x0 и x0 
плюс дельта где дельта равно 10 в степени минус 6 кодировались в латентное 
пространство. Измерялось расстояние между латентными векторами на каждом 
шаге траектории. Для истинной хаотической системы ожидается экспоненциальный 
рост расстояния с соотношением конечная точка делённое на начальная точка 
больше 5.0 что соответствует показателю Ляпунова λ приблизительно равному 
0.5 для логистической карты при r равном 3.99.

Таблица 4 представляет результаты по пяти независимым запускам с различными 
начальными условиями.

| Архитектура | Ratio (final/initial) | Интерпретация |
|-------------|----------------------|----------------|
| True chaos (baseline) | 105.7 ± 66.6 | Экспоненциальная дивергенция |
| Dense ReLU 64 | 0.90 ± 0.19 | Нет дивергенции |
| Dense ReLU 128 | 1.03 ± 0.05 | Стабильно |
| V4 K-Sparse Chaos | 1.02 ± 0.11 | Стабильно |

*Таблица 4: Тест на сохранение хаотической дивергенции (N=5 запусков). 
Для истинного хаоса ожидается ratio > 5.0.*

Все три архитектуры показали соотношение приблизительно равное единице что 
указывает на отсутствие экспоненциального роста расстояний. Различие с 
истинным хаосом составляет более двух порядков величины.

**Механизм потери temporal информации.** Feedforward автоэнкодер обрабатывает 
каждый кадр независимо без рекуррентных связей или временной памяти. Латентное 
представление z(t) зависит только от входного изображения в момент t и не 
содержит информации о предыдущих состояниях системы. Экспоненциальная 
дивергенция хаотических траекторий является временным эффектом где малые 
начальные различия накапливаются через последовательные итерации динамической 
системы. Автоэнкодер проецирует каждое состояние в латентное пространство 
независимо теряя информацию о временной эволюции.

**Что сохраняется.** Несмотря на потерю временной динамики V4 обеспечивает 
значительно большее абсолютное разделение траекторий в латентном пространстве. 
Средняя дистанция для V4 составляет 10.8 против 6.4 для Dense_64 что 
представляет улучшение в 1.7 раза. Это указывает на то что V4 сохраняет 
статические свойства хаоса: высокую дисперсию, разнообразие признаков, 
чувствительность к начальным условиям в пространстве представлений. 
Однако отсутствие экспоненциального роста подтверждает что временная динамика 
не сохраняется.

**Применимость метода.** Обнаруженное ограничение определяет границы 
применимости предложенного подхода. Метод подходит для задач где важны 
статические свойства латентного пространства: генерация псевдослучайных 
последовательностей, статическая интерпретируемость векторных представлений, 
обнаружение аномалий в данных с высокой энтропией. Метод не подходит для 
задач требующих сохранения временной динамики: прогнозирование временных 
рядов, оценка показателей Ляпунова, вычисления резервуаров. Для сохранения 
временной динамики необходимы архитектуры с явной временной памятью такие 
как LSTM, GRU или резервуарные вычисления.

**Масштабирование за пределы текущих доменов.** Метод протестирован на 
четырёх доменах: логистическая карта (dim~1), карта Хенона (dim~2), 
Лоренц-96 (dim~13-15) и MNIST (dim~10-20). Общие свойства (нулевые мёртвые 
нейроны, стабильная дисперсия) подтверждены на всех доменах (раздел 6.6). 
Однако масштабирование на данные с ещё более высокой внутренней размерностью, 
такие как CIFAR (dim~50) или векторные представления токенов в больших 
языковых моделях (dim~100+), остаётся непроверенным.

**Отсутствует сравнение с SOTA sparse methods.** Современные методы 
разреживания для языковых моделей такие как JumpReLU [8] представленный на 
ICLR 2025, Gated SAE [7] от NeurIPS 2024 и TopK SAE [9] от OpenAI разработаны 
для векторных представлений токенов с внутренней размерностью порядка 100 и 
выше. Прямое сравнение методологически сомнительно из-за фундаментального 
несоответствия доменов. Хаотические карты с внутренней размерностью 1-2 
представляют совершенно иной класс задач чем интерпретируемость языковых 
моделей. Тестирование предложенного метода на векторных представлениях 
языковых моделей или тестирование JumpReLU [8] и Gated SAE [7] на хаотических 
картах является необходимым условием для честного количественного сравнения 
методов.

**Reconstruction trade-off.** Метод даёт систематическое увеличение ошибки 
реконструкции на 8.2 процента по сравнению с гиперболическим тангенсом и 
на 6.2 процента по сравнению с плотной базовой архитектурой. Это приемлемый 
компромисс для приложений где высокая дисперсия латентного пространства 
является критическим требованием таких как генерация псевдослучайных 
последовательностей или обнаружение аномалий в данных с высокой энтропией. 
Однако для задач высокоточной реконструкции изображений или генеративных 
моделей где качество реконструкции является первостепенным такое ухудшение 
может быть неприемлемым. Периодичность синусоидальной компоненты приводит 
к необратимой потере информации которая не может быть компенсирована 
архитектурными модификациями без изменения самой функции активации.

**Механизм отбора по абсолютному значению.** Большинство разреженных 
автоэнкодеров для языковых моделей используют отбор по значению после 
применения функции ReLU что гарантирует строго неотрицательные активации 
и упрощает интерпретацию как весов признаков. Отбор по абсолютному значению 
был необходим для предложенной архитектуры так как функция sin(8x)+0.5·tanh(4x) 
даёт как положительные так и отрицательные значения. Это создаёт 
концептуальные трудности для интерпретации значения отрицательных активаций 
и делает результаты несопоставимыми с методами основанными только на 
положительных активациях. Разработка интерпретируемого семантического 
значения для отрицательных компонент латентного вектора является открытым 
вопросом для будущей работы.

## 8. СВЯЗАННЫЕ РАБОТЫ

[1] **Bricken et al., "Towards Monosemanticity: Decomposing Language Models With Dictionary Learning" (2023)**  
[Anthropic Transformer Circuits](https://transformer-circuits.pub/2023/monosemantic-features)  
- Основополагающая работа по разреженным словарям для интерпретируемости
- Продемонстрированы моносемантические признаки на малых моделях
- Фокус на статических данных, анализ хаоса отсутствует

[2] **Templeton et al., "Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet" (2024)**  
[Anthropic Transformer Circuits](https://transformer-circuits.pub/2024/scaling-monosemanticity/)  
- Масштабирование разреженных автоэнкодеров до производственных языковых моделей
- Разреженность 70–90% на векторных представлениях токенов
- Наша работа: аналогичные уровни разреженности, но на хаотических данных

[3] **Makhzani & Frey, "k-Sparse Autoencoders" (ICLR 2014)**  
[arXiv:1312.5663](https://arxiv.org/abs/1312.5663)  
- Оригинальная работа по k-sparse автоэнкодерам
- Предложен механизм отбора top-K активаций как метод разреживания
- Наша архитектура развивает этот подход для хаотических данных

[4] **Pathak et al., "Model-Free Prediction of Large Spatiotemporally Chaotic Systems from Data" (Physical Review Letters, 2018)**  
[Phys. Rev. Lett. 120, 024102](https://link.aps.org/doi/10.1103/PhysRevLett.120.024102)  
- Фундаментальная работа по предсказанию хаотических систем нейросетями
- Использованы echo state networks для воспроизведения аттракторов
- **Ключевое отличие:** отсутствует контроль разреженности — используются плотные представления

[5] **Margazoglou & Magri, "Stability analysis of chaotic systems from data" (Nonlinear Dynamics, 2023)**  
[Springer](https://link.springer.com/article/10.1007/s11071-023-08285-1) | [arXiv:2210.06167](https://arxiv.org/abs/2210.06167)  
- Вывод показателей Ляпунова из данных с помощью нейросетей
- Анализ устойчивости хаотических систем без явного моделирования уравнений

[6] **Özalp & Magri, "Inferring stability properties of chaotic systems on autoencoders' latent spaces" (NeurIPS ML4PS Workshop, 2024)**  
[arXiv:2410.18003](https://arxiv.org/abs/2410.18003)  
- Наиболее близкая к нашей работа: автоэнкодеры + хаос + показатели Ляпунова
- Исследование сохранения динамических свойств в латентном пространстве
- **Ключевое отличие:** используют плотные автоэнкодеры без контроля разреженности

[7] **Rajamanoharan et al., "Improving Sparse Decomposition of Language Model Activations with Gated Sparse Autoencoders" (NeurIPS 2024)**  
[NeurIPS Proceedings](https://papers.nips.cc/paper_files/paper/2024/hash/01772a8b0420baec00c4d59fe2fbace6-Abstract-Conference.html) | [arXiv:2404.16014](https://arxiv.org/abs/2404.16014)  
- Введён механизм вентилей для улучшения отбора признаков
- Улучшена проблема мёртвых нейронов на стандартных датасетах
- **Пробел:** не протестировано на данных с высокой дисперсией/хаотических входах

[8] **Rajamanoharan et al., "Jumping Ahead: Improving Reconstruction Fidelity with JumpReLU Sparse Autoencoders" (ICLR 2025)**  
[OpenReview](https://openreview.net/forum?id=mMPaQzgzAN) | [arXiv:2407.14435](https://arxiv.org/abs/2407.14435)  
- Наиболее актуальный SOTA для разреженных автоэнкодеров
- Достигается разреженность >90% при минимуме мёртвых нейронов
- **Критический пробел:** поведение на хаотических данных не исследовано

[9] **Gao et al., "Scaling and Evaluating Sparse Autoencoders" (ICLR 2025, Oral)**  
[OpenReview](https://openreview.net/forum?id=tcsZt9ZNKD) | [arXiv:2406.04093](https://arxiv.org/abs/2406.04093)  
- Введены TopK SAE и масштабирование до GPT-4 (16 млн латентных переменных, 40 млрд токенов)
- Систематическая оценка метрик качества разреженных представлений

[10] **Dunefsky et al., "Transcoders Find Interpretable LLM Feature Circuits" (NeurIPS 2024)**  
[arXiv:2406.11944](https://arxiv.org/abs/2406.11944)  
- Транскодеры как разреженные аппроксимации MLP-слоёв
- Обратное проектирование интерпретируемых цепочек в языковых моделях

